# 08 — Explanation Faithfulness/Stability, and the H6 Trust-Score Test

**Research purpose.** Two roadmap items are combined here because the second depends on the
first's output: (1) governing-prompt §17-18 (perturbation faithfulness, rank stability under
noise), still missing per §0.15's table; and (2) **H6** — "high predictive confidence does not
necessarily imply correct root-cause localization" — which the governing prompt explicitly singles
out as the hypothesis to test because it is "central to trustworthy AI" (§5), and which §64 frames
as the project's strongest candidate novel contribution: *does disagreement/low-confidence among
the framework's own signals predict when its RCA output is wrong?* This is tested here, not
assumed — per §66/§78, the result is reported whichever way it comes out.

**Input.** `src/data.py`'s split; a LightGBM+SHAP pair and a Transformer trained fresh (as in
notebooks `02`/`04`); `results/rca_localization_results.csv` (notebook `04`'s per-event Hit@k).

**Output.** `results/faithfulness_and_trust_results.csv` (per-event: faithfulness/stability/
confidence signals + RCA correctness) and the Critical Trustworthiness Matrix (governing prompt
§53) as a printed table.

**Paper relevance.** Directly answers RQ4 (explainability reliability) and provides the empirical
test of H6/§64's trust-score research question — the single most-repeated priority in the governing
prompt (§5, §24, §63-65, §78).

**Power caveat, stated up front, not buried at the end**: this machine has 8 events. Every
statistic below is descriptive of `machine-1-1`, not a confirmatory population-level test — this is
restated at the point of computing anything, consistent with §0.12/§0.14 of the main notebook's
audit.


In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import lightgbm as lgb
import torch, torch.nn as nn
import shap
from scipy.stats import spearmanr, kendalltau

from src.data import load_machine, build_split, WINDOW, HORIZON
from src.registry import append_row

FEAT = 38
m = load_machine("machine-1-1")
sd = build_split(m)

pos, neg = sd.train_y_full.sum(), len(sd.train_y_full) - sd.train_y_full.sum()
SPW = neg / max(pos, 1)

gbm = lgb.LGBMClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                          scale_pos_weight=SPW, verbose=-1, random_state=0)
gbm.fit(sd.train_tab_full, sd.train_y_full)
explainer = shap.TreeExplainer(gbm)
tab_by_block = {"train": sd.sup_tab, "eval": sd.ev_tab, "calib": sd.cal_tab}
raw_by_block = {"train": sd.sup_raw, "eval": sd.ev_raw, "calib": sd.cal_raw}


class TinyTransformer(nn.Module):
    # Identical architecture to the main notebook's Section 6 model.
    def __init__(self, feat=FEAT, d_model=32, nhead=4, layers=2, window=WINDOW):
        super().__init__()
        self.proj = nn.Linear(feat, d_model)
        self.pos = nn.Parameter(torch.randn(1, window, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=64,
                                                batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(d_model, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        h = self.encoder(self.proj(x) + self.pos)
        return self.head(h.mean(dim=1)).squeeze(-1)


torch.manual_seed(0)
model = TinyTransformer()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([SPW], dtype=torch.float32))
Xtr = torch.tensor(sd.train_raw_full, dtype=torch.float32)
ytr = torch.tensor(sd.train_y_full, dtype=torch.float32)
for _ in range(6):
    perm = torch.randperm(len(Xtr))
    model.train()
    for i in range(0, len(Xtr), 256):
        idx = perm[i:i + 256]
        opt.zero_grad()
        loss = lossf(model(Xtr[idx]), ytr[idx])
        loss.backward(); opt.step()
model.eval()

# split-conformal threshold from CALIB (identical procedure to the main notebook's Section 7)
with torch.no_grad():
    p_cal = torch.sigmoid(model(torch.tensor(sd.cal_raw, dtype=torch.float32))).numpy()
alpha = 0.10
p_true_cal = np.where(sd.cal_y == 1, p_cal, 1 - p_cal)
qhat = np.quantile(1 - p_true_cal, np.ceil((len(p_cal) + 1) * (1 - alpha)) / len(p_cal), method="higher")
print(f"Trained LightGBM+SHAP, Transformer, and conformal qhat={qhat:.3f}. Ready.")

Trained LightGBM+SHAP, Transformer, and conformal qhat=0.985. Ready.


## Explanation stability under small input perturbation (§18)

For each event's pre-onset window, add small Gaussian noise (`σ=0.05` in already-standardized
units — roughly 5% of one standard deviation) to the raw window, recompute the tabular features and
the SHAP ranking, and compare to the unperturbed ranking via Spearman ρ, Kendall τ, and Top-5
overlap. Averaged over 20 noise draws per event.


In [2]:
def tab_from_raw(window_raw: np.ndarray) -> np.ndarray:
    # Rebuild the 114-dim [mean,std,last] tabular vector from a perturbed raw (W,38) window --
    # mirrors src/data.py's make_windows so a perturbed SHAP ranking is computed the same way.
    return np.concatenate([window_raw.mean(0), window_raw.std(0), window_raw[-1]])[None, :]


def shap_full_ranking_from_tab(tab_row: np.ndarray) -> np.ndarray:
    sv = explainer.shap_values(tab_row)
    sv = np.asarray(sv[1] if isinstance(sv, list) else sv).reshape(-1)
    return np.abs(sv[:FEAT]) + np.abs(sv[FEAT:2*FEAT]) + np.abs(sv[2*FEAT:3*FEAT])


rng = np.random.default_rng(0)
stability_rows = {}
for (s, e, dims_gt) in m.segments:
    t_pre = s - 1
    if t_pre not in sd.block_index:
        continue
    name, row = sd.block_index[t_pre]
    window_raw = raw_by_block[name][row]  # (20, 38)
    base_agg = shap_full_ranking_from_tab(tab_from_raw(window_raw))
    base_rank = np.argsort(-base_agg)

    rhos, taus, overlaps = [], [], []
    for _ in range(20):
        noisy = window_raw + rng.normal(0, 0.05, size=window_raw.shape).astype(np.float32)
        agg = shap_full_ranking_from_tab(tab_from_raw(noisy))
        rank = np.argsort(-agg)
        rho, _ = spearmanr(base_agg, agg)
        tau, _ = kendalltau(base_agg, agg)
        rhos.append(rho); taus.append(tau)
        overlaps.append(len(set(base_rank[:5]) & set(rank[:5])) / 5)

    stability_rows[f"{s}-{e}"] = dict(block=name, spearman_rho=np.mean(rhos),
                                       kendall_tau=np.mean(taus), top5_overlap=np.mean(overlaps),
                                       attribution_entropy=-(base_agg / base_agg.sum() *
                                                              np.log(base_agg / base_agg.sum() + 1e-12)).sum() / np.log(FEAT))

stability_df = pd.DataFrame(stability_rows).T
display(stability_df)

,block,spearman_rho,kendall_tau,top5_overlap,attribution_entropy
15849-16368,train,0.989926,0.954692,1.0,0.710797
16963-17517,train,0.995794,0.968035,0.98,0.717605
18071-18528,train,0.99804,0.982111,1.0,0.688014
19367-20088,eval,0.992205,0.957625,1.0,0.711142
20786-21195,eval,0.995761,0.971554,0.85,0.70433
24679-24682,calib,0.99008,0.961144,1.0,0.70941
26114-26116,calib,0.994396,0.976246,1.0,0.695476
27554-27556,calib,0.992348,0.966862,1.0,0.694646


**Reading this**: `spearman_rho`/`kendall_tau` close to 1.0 mean the ranking barely moves
under 5%-noise perturbation (stable); `top5_overlap` is the fraction of the same dimensions staying
in the top-5. `attribution_entropy` (normalized 0-1) is a second, perturbation-free confidence
proxy: low entropy means SHAP concentrates its attribution on a few dimensions (a "confident,"
peaked explanation); high entropy means it spreads attribution thinly across many dimensions.


## Explanation faithfulness: deletion test (§17)

For each event, zero out (impute the standardized training mean, i.e. 0.0) the top-5 SHAP-ranked
dimensions' `mean/std/last` tabular entries and measure the resulting drop in `gbm`'s predicted
probability, compared against deleting 5 *random* dimensions (200 repeats, averaged). A faithful
explanation should cause a **larger** probability drop than a random deletion of the same size.


In [3]:
def delete_dims(tab_row: np.ndarray, dims_1indexed: list[int]) -> np.ndarray:
    out = tab_row.copy()
    for d in dims_1indexed:
        i = d - 1
        out[0, i] = 0.0
        out[0, FEAT + i] = 0.0
        out[0, 2 * FEAT + i] = 0.0
    return out


faith_rows = {}
for (s, e, dims_gt) in m.segments:
    t_pre = s - 1
    if t_pre not in sd.block_index:
        continue
    name, row = sd.block_index[t_pre]
    tab_row = tab_by_block[name][row:row + 1]
    p_base = gbm.predict_proba(tab_row)[0, 1]

    agg = shap_full_ranking_from_tab(tab_row)
    top5 = (np.argsort(-agg)[:5] + 1).tolist()
    p_top5_deleted = gbm.predict_proba(delete_dims(tab_row, top5))[0, 1]

    random_drops = []
    for _ in range(200):
        rand5 = (rng.choice(FEAT, size=5, replace=False) + 1).tolist()
        p_rand = gbm.predict_proba(delete_dims(tab_row, rand5))[0, 1]
        random_drops.append(abs(p_base - p_rand))

    faith_rows[f"{s}-{e}"] = dict(
        p_base=p_base, top5_drop=abs(p_base - p_top5_deleted),
        random5_drop_mean=np.mean(random_drops), random5_drop_std=np.std(random_drops),
        faithful=abs(p_base - p_top5_deleted) > np.mean(random_drops),
    )

faith_df = pd.DataFrame(faith_rows).T
faith_df["faithful"] = faith_df["faithful"].astype(bool)  # dict-of-dicts + .T can upcast mixed rows to object dtype
display(faith_df)
n_faithful = sum(bool(v) for v in faith_df.faithful)
print(f"\nTop-5 SHAP deletion beats random-5 deletion in {n_faithful}/{len(faith_df)} events.")

,p_base,top5_drop,random5_drop_mean,random5_drop_std,faithful
15849-16368,0.999918,0.999914,0.193585,0.328778,True
16963-17517,0.999921,0.999899,0.186568,0.328216,True
18071-18528,0.999952,0.999949,0.191637,0.328694,True
19367-20088,0.629426,0.629411,0.4074,0.250436,True
20786-21195,0.209129,0.208399,0.199841,0.175621,True
24679-24682,0.517275,0.517274,0.320315,0.205862,True
26114-26116,0.941049,0.941027,0.447952,0.390599,True
27554-27556,0.999606,0.999603,0.250073,0.377158,True



Top-5 SHAP deletion beats random-5 deletion in 8/8 events.


**Reading this**: `faithful=True` for all 8 events looks like a clean confirmation, but the
margin varies enormously (e.g. `20786-21195`: top5_drop 0.208 vs. random-5 mean 0.200, barely
different, vs. `15849-16368`: 0.9999 vs. 0.194, a huge margin) — and three of the eight events start
from `p_base > 0.999`, i.e. `gbm` is already at its output ceiling before any deletion, so "the top-5
deletion drops probability more than random" is close to a floor effect (there is very little room
for random deletion to move a probability already pinned near 1.0) rather than uniformly strong
evidence of faithfulness. The `20786-21195` case — the same event notebook `04` found SHAP's
attribution weakest on — also shows the weakest faithfulness margin here, which is at least
internally consistent rather than contradictory.


## H6 and the Critical Trustworthiness Matrix (governing prompt §52-53, §64)

Per-event signals assembled: `p_hat` (Transformer probability at `t_pre`), the conformal set label
(`{degrading}` = confident-and-correct here, since by construction every event's `t_pre` has true
label 1; `{normal, degrading}` = ambiguous; `{normal}` would be confident-and-wrong), the
attribution-entropy and stability signals from above, and `RCA_hit@1` reloaded from notebook `04`'s
saved results. **H6 asks**: does `p_hat`/conformal confidence predict `RCA_hit@1`? Tested, not
assumed.


In [4]:
def conformal_label(p1: float, qhat: float) -> str:
    keep0, keep1 = (p1 <= qhat), ((1 - p1) <= qhat)
    if keep0 and keep1: return "ambiguous"
    if keep1: return "confident_degrading"
    if keep0: return "confident_normal"
    return "empty"


rca_df = pd.read_csv("../results/rca_localization_results.csv")
hit1 = (rca_df[(rca_df.method == "shap") & (rca_df.metric == "hit@1")]
        .set_index("segment")["value"].astype(bool))

matrix_rows = []
for (s, e, dims_gt) in m.segments:
    t_pre = s - 1
    if t_pre not in sd.block_index:
        continue
    name, row = sd.block_index[t_pre]
    x = torch.tensor(raw_by_block[name][row:row + 1], dtype=torch.float32)
    with torch.no_grad():
        p_hat = float(torch.sigmoid(model(x)).item())
    seg = f"{s}-{e}"
    matrix_rows.append(dict(
        segment=seg, block=name, p_hat=p_hat, conformal=conformal_label(p_hat, qhat),
        prediction_correct=p_hat >= 0.5,  # true label is always 1 at t_pre by target construction
        rca_hit1=bool(hit1.get(seg, False)),
        attribution_entropy=stability_df.loc[seg, "attribution_entropy"],
        stability_rho=stability_df.loc[seg, "spearman_rho"],
        faithfulness_margin=faith_df.loc[seg, "top5_drop"] - faith_df.loc[seg, "random5_drop_mean"],
    ))

trust_df = pd.DataFrame(matrix_rows)
trust_df.to_csv("../results/faithfulness_and_trust_results.csv", index=False)
display(trust_df)

,segment,block,p_hat,conformal,prediction_correct,rca_hit1,attribution_entropy,stability_rho,faithfulness_margin
0,15849-16368,train,0.996010,confident_degrading,True,True,0.710797,0.989926,0.806329
1,16963-17517,train,0.995641,confident_degrading,True,True,0.717605,0.995794,0.813330
2,18071-18528,train,0.995002,confident_degrading,True,False,0.688014,0.998040,0.808313
3,19367-20088,eval,0.990731,confident_degrading,True,True,0.711142,0.992205,0.222011
4,20786-21195,eval,0.976833,ambiguous,True,False,0.704330,0.995761,0.008558
5,24679-24682,calib,0.996147,confident_degrading,True,True,0.709410,0.990080,0.196959
6,26114-26116,calib,0.994992,confident_degrading,True,False,0.695476,0.994396,0.493075
7,27554-27556,calib,0.995314,confident_degrading,True,False,0.694646,0.992348,0.749530


In [5]:
print("Critical Trustworthiness Matrix (governing prompt Section 53):\n")
ctab = pd.crosstab(trust_df.prediction_correct.map({True: "prediction_correct", False: "prediction_wrong"}),
                    trust_df.rca_hit1.map({True: "RCA_correct(hit@1)", False: "RCA_wrong(hit@1)"}))
display(ctab)

print("\nBy conformal confidence level vs RCA correctness:\n")
ctab2 = pd.crosstab(trust_df.conformal, trust_df.rca_hit1.map({True: "RCA_correct", False: "RCA_wrong"}))
display(ctab2)

Critical Trustworthiness Matrix (governing prompt Section 53):



rca_hit1,RCA_correct(hit@1),RCA_wrong(hit@1)
prediction_correct,,
prediction_correct,4,4



By conformal confidence level vs RCA correctness:



rca_hit1,RCA_correct,RCA_wrong
conformal,,
ambiguous,0,1
confident_degrading,4,3


In [6]:
print("H6 test -- descriptive only, n=8, no confirmatory claim (Section 0.12/0.14 power caveat):\n")
for signal in ["p_hat", "attribution_entropy", "stability_rho", "faithfulness_margin"]:
    hit = trust_df.loc[trust_df.rca_hit1, signal]
    miss = trust_df.loc[~trust_df.rca_hit1, signal]
    print(f"{signal:22s}: RCA-correct events mean={hit.mean():.3f} (n={len(hit)})   "
          f"RCA-wrong events mean={miss.mean():.3f} (n={len(miss)})")

append_row("../results/experiment_registry.csv", dict(
    experiment="08_explanation_faithfulness_and_trust", model="gbm+transformer", machine="machine-1-1",
    n_events=len(trust_df), n_rca_correct=int(trust_df.rca_hit1.sum()),
    n_prediction_correct=int(trust_df.prediction_correct.sum()),
))

H6 test -- descriptive only, n=8, no confirmatory claim (Section 0.12/0.14 power caveat):

p_hat                 : RCA-correct events mean=0.995 (n=4)   RCA-wrong events mean=0.991 (n=4)
attribution_entropy   : RCA-correct events mean=0.712 (n=4)   RCA-wrong events mean=0.696 (n=4)
stability_rho         : RCA-correct events mean=0.992 (n=4)   RCA-wrong events mean=0.995 (n=4)
faithfulness_margin   : RCA-correct events mean=0.510 (n=4)   RCA-wrong events mean=0.515 (n=4)


**Reading H6 — this is the cleanest result in the project so far, and it supports H6.** As
anticipated (the Transformer's own lead-time analysis in the main notebook already showed it ramps
up sharply right at `t_pre`), `prediction_correct` is `True` for all 8 events — the Critical
Trustworthiness Matrix collapses to exactly the two categories §53 names by name: **"ideal"
(4 events: prediction correct + RCA correct) vs. "dangerous diagnostic error" (4 events: prediction
correct + RCA wrong)**, with zero "detection failure" or "total failure" cases on this machine.

Restricting to those 8 always-confident-about-degradation events, **none of the four candidate
confidence/quality signals separate the 4 RCA-correct events from the 4 RCA-wrong ones by any
meaningful margin**:

| Signal | RCA-correct mean (n=4) | RCA-wrong mean (n=4) | Difference |
|---|---|---|---|
| `p_hat` (Transformer probability) | 0.995 | 0.991 | 0.004 |
| `attribution_entropy` (SHAP spread) | 0.712 | 0.696 | 0.016 |
| `stability_rho` (rank stability under noise) | 0.992 | 0.995 | -0.003 |
| `faithfulness_margin` (deletion test) | 0.510 | 0.515 | -0.005 |

Every difference is within noise, and two of the four (`stability_rho`, `faithfulness_margin`) point
in the *wrong* direction (slightly higher, not lower, on the wrong-RCA events). The conformal layer
shows the same pattern: the single `ambiguous` event happens to be an RCA-wrong case, but 3 of the
remaining 4 RCA-wrong events are still labeled `confident_degrading` — confidence does not flag them.

**This is a direct, if small-sample, demonstration of H6**: on this machine's evidence, the model
can be maximally confident that a problem exists, produce a stable, high-entropy-free, "faithful"-
by-deletion-test explanation, and still be wrong about *which dimension* is responsible — with
**no signal available in this notebook's own outputs** distinguishing that case from a correct one.
The practical implication for §64's trust-score idea is negative-but-informative: naively combining
`p_hat`, attribution entropy, and stability into one composite `Trust(e)` would **not**, on this
evidence, separate correct from incorrect RCA — building that score would need either a different
signal not tested here (e.g. Section 3's causal-evidence agreement, or cross-machine variance) or
more data to detect a smaller effect these 4-vs-4 groups are too small to resolve. Per §66, this
negative result is kept as-is rather than reframed as a partial success.
